# VisionTSRAR 演示 Notebook

本 Notebook 展示 VisionTSRAR 的完整预测流程：
1. 数据准备：加载时间序列数据
2. 模型初始化：创建 VisionTSRAR 实例并加载预训练权重
3. 模型配置：设置时序参数（上下文长度、预测长度、周期性等）
4. 前向推理：运行完整的6步流水线
5. 可视化：输入图像、重建图像、时序预测对比图

In [ ]:
import os
import sys
import torch
import numpy as np
import matplotlib.pyplot as plt

# 将项目根目录加入搜索路径
project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# 设备自动检测：CUDA > MPS > CPU
if torch.cuda.is_available():
    device = 'cuda'
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'
print(f'使用设备: {device}')
print(f'PyTorch 版本: {torch.__version__}')

## 1. 数据准备

创建一个示例时间序列数据（正弦波 + 噪声），模拟 ETTh1 格式。

In [ ]:
# 生成示例时间序列数据
np.random.seed(42)
seq_len = 96      # 输入序列长度
pred_len = 96     # 预测序列长度
nvars = 7         # 变量数（ETTh1有7个特征）

# 创建带有日周期性的多变量时间序列
t = np.linspace(0, 4 * np.pi, seq_len + pred_len)
x_full = np.stack([
    np.sin(t * (i + 1) / 3.0) + 0.1 * np.random.randn(len(t))
    for i in range(nvars)
], axis=-1)  # shape: [seq_len+pred_len, nvars]

x_enc = x_full[:seq_len]       # 输入序列 [seq_len, nvars]
x_true = x_full[seq_len:]      # 真实未来值 [pred_len, nvars]

# 转为 PyTorch tensor
x_enc_tensor = torch.FloatTensor(x_enc).unsqueeze(0)  # [1, seq_len, nvars]
if device != 'cpu':
    x_enc_tensor = x_enc_tensor.to(device)

print(f'输入数据形状: {x_enc_tensor.shape}')
print(f'输入数据范围: [{x_enc.min():.3f}, {x_enc.max():.3f}]')

## 2. 模型初始化

创建 VisionTSRAR 模型实例。首次运行时会自动从 HuggingFace 下载预训练权重（约800MB）。

In [ ]:
from visiontsrar import VisionTSRAR

# 创建模型
model = VisionTSRAR(
    arch='rar_l_0.3b',        # RAR架构：0.3B参数
    finetune_type='ln',        # 微调策略：仅LayerNorm
    ckpt_dir='../ckpt/',       # 权重目录
    load_ckpt=True,            # 自动加载预训练权重
    num_inference_steps=88,    # RAR推理步数
    position_order='raster',   # 光栅顺序（适合时序预测）
)

if device != 'cpu':
    model = model.to(device)

model.eval()
print('模型初始化完成！')
print(f'模型参数量: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M')
print(f'可训练参数量: {sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6:.1f}M')

## 3. 模型配置

设置时序预测的关键参数：上下文长度、预测长度、周期性等。

In [ ]:
# 配置模型参数
model.update_config(
    context_len=seq_len,      # 上下文长度
    pred_len=pred_len,        # 预测长度
    periodicity=24,           # 周期性（小时级数据的日周期）
    norm_const=0.4,           # 归一化常数
    align_const=0.4,          # 对齐常数（输入占图像比例）
)

print(f'配置完成：context_len={seq_len}, pred_len={pred_len}, periodicity=24')

## 4. 前向推理

运行完整的6步流水线：Normalization → Segmentation → Render & Alignment → RAR Reconstruction → Forecasting → Denormalization

In [ ]:
with torch.no_grad():
    prediction = model(x_enc_tensor)  # [1, pred_len, nvars]

# 移回CPU用于可视化
pred_np = prediction.cpu().numpy()[0]  # [pred_len, nvars]

print(f'预测输出形状: {prediction.shape}')
print(f'预测值范围: [{pred_np.min():.3f}, {pred_np.max():.3f}]')

## 5. 可视化

展示输入时序、真实未来值、预测值的对比图。

In [ ]:
# 绘制预测结果对比图
fig, axes = plt.subplots(nvars, 1, figsize=(14, 2.5 * nvars), sharex=True)

t_input = np.arange(seq_len)
t_pred = np.arange(seq_len, seq_len + pred_len)

for i in range(nvars):
    ax = axes[i] if nvars > 1 else axes
    # 输入序列
    ax.plot(t_input, x_enc[:, i], 'b-', linewidth=1.5, label='输入')
    # 真实未来值
    ax.plot(t_pred, x_true[:, i], 'g-', linewidth=1.5, label='真实值')
    # 预测值
    ax.plot(t_pred, pred_np[:, i], 'r--', linewidth=1.5, label='预测值')
    # 分割线
    ax.axvline(x=seq_len, color='gray', linestyle=':', alpha=0.5)
    ax.set_ylabel(f'Var {i+1}')
    if i == 0:
        ax.legend(loc='upper left')

axes[-1].set_xlabel('Time Step')
plt.suptitle('VisionTSRAR 时序预测结果', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 6. 评估指标

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error

mse = mean_squared_error(x_true.flatten(), pred_np.flatten())
mae = mean_absolute_error(x_true.flatten(), pred_np.flatten())

print(f'MSE: {mse:.4f}')
print(f'MAE: {mae:.4f}')
print()
print('注意：由于使用的是随机生成的示例数据，以上指标仅供参考。')
print('如需评估真实性能，请使用 long_term_tsf/ 实验框架在标准数据集上测试。')